## Creating Silver Delta Tables and Transforming Using Spark

Transforming bronze dimensional data into clean, deduplicated silver tables using PySpark:
* **slv_brands** - Brand master data
* **slv_category** - Product categories
* **slv_products** - Product catalog
* **slv_customers** - Customer master data
* **slv_calendar** - Calendar dimension

**Key Features**:
- PySpark transformations with window functions
- Delta Lake MERGE for upserts (idempotent)
- Data quality transformations and deduplication

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# Configuration
dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")
catalog_name = dbutils.widgets.get("catalog_name")

print(f"Using catalog: {catalog_name}")

In [0]:
# Brands - Silver Transformation

df_brands = spark.table(f"{catalog_name}.bronze.brz_brands")

# Define window for deduplication
window_brands = Window.partitionBy(
    F.regexp_replace(F.trim(F.col("brand_code")), '[^A-Za-z0-9]', '')
).orderBy(F.col("_ingested_at").desc())

# Transformations
df_brands_clean = (df_brands
    .withColumn("brand_code", F.regexp_replace(F.trim(F.col("brand_code")), '[^A-Za-z0-9]', ''))
    .withColumn("brand_name", F.trim(F.col("brand_name")))
    .withColumn("category_code", 
        F.when(F.col("category_code") == "GROCERY", "GRCY")
         .when(F.col("category_code") == "BOOKS", "BKS")
         .when(F.col("category_code") == "TOYS", "TOY")
         .otherwise(F.col("category_code"))
    )
    .withColumn("rn", F.row_number().over(window_brands))
    .filter(F.col("rn") == 1)
    .drop("rn", "_source_file")
)

# Merge into silver table
if spark.catalog.tableExists(f"{catalog_name}.silver.slv_brands"):
    delta_brands = DeltaTable.forName(spark, f"{catalog_name}.silver.slv_brands")
    
    (delta_brands.alias("target")
        .merge(
            df_brands_clean.alias("source"),
            "target.brand_code = source.brand_code"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    df_brands_clean.write.format("delta").mode("overwrite").saveAsTable(f"{catalog_name}.silver.slv_brands")

In [0]:
# Category - Silver Transformation

df_category = spark.table(f"{catalog_name}.bronze.brz_category")

# Define window for deduplication
window_category = Window.partitionBy(
    F.upper(F.trim(F.col("category_code")))
).orderBy(F.col("_ingested_at").desc())

# Transformations
df_category_clean = (df_category
    .withColumn("category_code", F.upper(F.trim(F.col("category_code"))))
    .withColumn("category_name", F.trim(F.col("category_name")))
    .withColumn("rn", F.row_number().over(window_category))
    .filter(F.col("rn") == 1)
    .drop("rn", "_source_file")
)

# Merge into silver table
if spark.catalog.tableExists(f"{catalog_name}.silver.slv_category"):
    delta_category = DeltaTable.forName(spark, f"{catalog_name}.silver.slv_category")
    
    (delta_category.alias("target")
        .merge(
            df_category_clean.alias("source"),
            "target.category_code = source.category_code"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    df_category_clean.write.format("delta").mode("overwrite").saveAsTable(f"{catalog_name}.silver.slv_category")

In [0]:
# Products - Silver Transformation

df_products = spark.table(f"{catalog_name}.bronze.brz_products")

# Define window for deduplication
window_products = Window.partitionBy(
    F.trim(F.col("product_id"))
).orderBy(F.col("_ingested_at").desc())

# Transformations
df_products_clean = (df_products
    .withColumn("product_id", F.trim(F.col("product_id")))
    .withColumn("sku", F.upper(F.trim(F.col("sku"))))
    .withColumn("category_code", F.upper(F.trim(F.col("category_code"))))
    .withColumn("brand_code", F.upper(F.trim(F.col("brand_code"))))
    .withColumn("color", F.initcap(F.trim(F.col("color"))))
    .withColumn("size", F.trim(F.col("size")))
    .withColumn("material",
        F.when(F.lower(F.trim(F.col("material"))) == "coton", "Cotton")
         .when(F.lower(F.trim(F.col("material"))) == "ruber", "Rubber")
         .otherwise(F.initcap(F.trim(F.col("material"))))
    )
    .withColumn("weight_grams", 
        F.regexp_replace(F.col("weight_grams"), '[^0-9.]', '').cast("float")
    )
    .withColumn("length_cm",
        F.round(F.regexp_replace(F.col("length_cm"), ',', '.').cast("float"), 2)
    )
    .withColumn("width_cm", F.round(F.col("width_cm").cast("float"), 2))
    .withColumn("height_cm", F.round(F.col("height_cm").cast("float"), 2))
    .withColumn("rating_count",
        F.when(F.col("rating_count").isNotNull(), F.abs(F.col("rating_count").cast("int")))
         .otherwise(0)
    )
    .withColumn("rn", F.row_number().over(window_products))
    .filter(F.col("rn") == 1)
    .drop("rn", "_source_file")
)

# Merge into silver table
if spark.catalog.tableExists(f"{catalog_name}.silver.slv_products"):
    delta_products = DeltaTable.forName(spark, f"{catalog_name}.silver.slv_products")
    
    (delta_products.alias("target")
        .merge(
            df_products_clean.alias("source"),
            "target.product_id = source.product_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    df_products_clean.write.format("delta").mode("overwrite").saveAsTable(f"{catalog_name}.silver.slv_products")

print(f"✓ slv_products: {df_products_clean.count():,} rows")

In [0]:
# Customers - Silver Transformation

df_customers = spark.table(f"{catalog_name}.bronze.brz_customers")

# Define window for deduplication
window_customers = Window.partitionBy(
    F.trim(F.col("customer_id"))
).orderBy(F.col("_ingested_at").desc())

# Transformations
df_customers_clean = (df_customers
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("phone").isNotNull())
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("phone", F.trim(F.col("phone")))
    .withColumn("country", F.upper(F.trim(F.col("country"))))
    .withColumn("rn", F.row_number().over(window_customers))
    .filter(F.col("rn") == 1)
    .select("customer_id", "phone", "country", "_ingested_at")
)

# Merge into silver table
if spark.catalog.tableExists(f"{catalog_name}.silver.slv_customers"):
    delta_customers = DeltaTable.forName(spark, f"{catalog_name}.silver.slv_customers")
    
    (delta_customers.alias("target")
        .merge(
            df_customers_clean.alias("source"),
            "target.customer_id = source.customer_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    df_customers_clean.write.format("delta").mode("overwrite").saveAsTable(f"{catalog_name}.silver.slv_customers")

In [0]:
# Calendar - Silver Transformation

df_calendar = spark.table(f"{catalog_name}.bronze.brz_date")

# Define window for deduplication
window_calendar = Window.partitionBy(
    F.col("date").cast("date")
).orderBy(F.col("_ingested_at").desc())

# Transformations
df_calendar_clean = (df_calendar
    .filter(F.col("date").isNotNull())
    .withColumn("date", F.col("date").cast("date"))
    .withColumn("year", F.col("year").cast("int"))
    .withColumn("day_name", F.initcap(F.trim(F.col("day_name"))))
    .withColumn("quarter", F.concat(F.lit("Q"), F.col("quarter").cast("int"), F.lit("-"), F.col("year").cast("int")))
    .withColumn("week_of_year",
        F.when(F.col("week_of_year").cast("int") < 1, F.weekofyear(F.col("date")))
         .otherwise(F.col("week_of_year").cast("int"))
    )
    .withColumn("rn", F.row_number().over(window_calendar))
    .filter(F.col("rn") == 1)
    .drop("rn", "_source_file")
)

# Merge into silver table
if spark.catalog.tableExists(f"{catalog_name}.silver.slv_calendar"):
    delta_calendar = DeltaTable.forName(spark, f"{catalog_name}.silver.slv_calendar")
    
    (delta_calendar.alias("target")
        .merge(
            df_calendar_clean.alias("source"),
            "target.date = source.date"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    df_calendar_clean.write.format("delta").mode("overwrite").saveAsTable(f"{catalog_name}.silver.slv_calendar")

In [0]:
# Dim_silver summary
print("Silver Dimensional Tables Summary")

tables = ["slv_brands", "slv_category", "slv_products", "slv_customers", "slv_calendar"]

for table in tables:
    count = spark.table(f"{catalog_name}.silver.{table}").count()
    print(f"{table:25} : {count:,} rows")

print("\n All silver dimensional tables processed successfully!")